In [ ]:
import os
import subprocess
from pathlib import Path

S3_ACCESS_KEY = ""
S3_SECRET_KEY = ""
S3_BUCKET = "s3://my-bucket/data"
S3_ENDPOINT_URL = ""  # leave empty for AWS; set for MinIO or other S3-compatible stores

# Walk up from cwd to find the git repo root
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / ".git").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

In [ ]:
def configure_s3_remote():
    def dvc(*args):
        subprocess.run(["dvc"] + list(args), cwd=REPO_ROOT, check=True)

    dvc("remote", "add", "-d", "-f", "myremote", S3_BUCKET)
    dvc("remote", "modify", "myremote", "access_key_id", S3_ACCESS_KEY)
    dvc("remote", "modify", "myremote", "secret_access_key", S3_SECRET_KEY)
    if S3_ENDPOINT_URL:
        dvc("remote", "modify", "myremote", "endpointurl", S3_ENDPOINT_URL)

In [ ]:
def pull():
    env = os.environ.copy()
    if S3_ACCESS_KEY:
        env["AWS_ACCESS_KEY_ID"] = S3_ACCESS_KEY
    if S3_SECRET_KEY:
        env["AWS_SECRET_ACCESS_KEY"] = S3_SECRET_KEY

    subprocess.run(["dvc", "pull"], cwd=REPO_ROOT, check=True, env=env)
    print("DVC pull complete.")

In [ ]:
configure_s3_remote()
pull()